# C2.4 — Advanced AI Workflows: LCEL · LlamaIndex · MCP · Multi-Agent · Prompt Versioning

**Domain focus:** Finance, Education, Healthcare, E-commerce — examples throughout are drawn from these industries.

| Section | Topic |
|---------|-------|
| 1 | LangChain LCEL — multi-step chains and memory |
| 2 | LlamaIndex — data connectors, index types, query engines |
| 3 | MCP — Model Context Protocol with a local server |
| 4 | Multi-agent orchestration — coordinator + specialist |
| 5 | Prompt versioning and A/B testing |
| 6 | Lab — end-to-end AI workflow |

## Setup

In [ ]:
# Install required packages (run once)
# !pip install langchain-core langchain-anthropic llama-index-core llama-index-llms-anthropic
# !pip install llama-index-embeddings-huggingface mcp anthropic

In [ ]:
import os, json, uuid, time, asyncio, subprocess, sys, textwrap
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field
from typing import Optional

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
print('API key loaded:', bool(ANTHROPIC_API_KEY))

---
## Section 1 · LangChain LCEL (LangChain Expression Language)

<small>
LCEL is the **declarative composition layer** introduced in LangChain v0.1+.
It replaces the legacy `LLMChain` / `SequentialChain` classes with a simple pipe syntax.

```
# Legacy (avoid)
chain = LLMChain(llm=llm, prompt=prompt)

# LCEL (current)
chain = prompt | llm | parser
```

**Why LCEL?**
- Streaming and async work out of the box
- Parallel branches with `RunnableParallel`
- Explicit, composable memory via `RunnableWithMessageHistory`
- No hidden state — every step is a Runnable you can inspect
</small>

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_anthropic import ChatAnthropic

import sys
sys.path.append('Track2/Functions')

llm = ChatAnthropic(
    model='claude-haiku-4-5-20251001',
    temperature=0.3,
    anthropic_api_key=ANTHROPIC_API_KEY
)
parser = StrOutputParser()

print('LLM ready')

### 1.1 Basic LCEL Chain — Finance: Earnings Report Analysis

<small>A simple three-step chain: prompt → LLM → parser.
The finance scenario: an analyst pastes a raw earnings paragraph; the chain returns a structured summary.
</small>

In [ ]:
earnings_prompt = ChatPromptTemplate.from_template(
    'You are a financial analyst assistant.\n'
    'Analyze the following earnings report excerpt and return:\n'
    '1. Revenue trend (up/down/flat)\n'
    '2. Key risk factors (max 2 bullet points)\n'
    '3. One-line investment signal\n\n'
    'Report:\n{report}'
)

earnings_chain = earnings_prompt | llm | parser

sample_report = '''
Q3 2024: Revenue rose 12% YoY to $4.8B, beating consensus by 3%.
Operating margin compressed 80bps to 18.2% due to higher cloud infrastructure costs.
Guidance for Q4 raised to $5.0–5.1B. Management flagged FX headwinds in EMEA.
'''

result = earnings_chain.invoke({'report': sample_report})
print(result)

### 1.2 Multi-Step Chain with `RunnablePassthrough`

<small>
`RunnablePassthrough` forwards the input unchanged while other keys are added.
Pattern: `{'context': retriever, 'question': RunnablePassthrough()} | prompt | llm`

**Education scenario:** A student submits an essay; the chain grades it, then writes personalised feedback.
</small>

In [ ]:
grade_prompt = ChatPromptTemplate.from_template(
    'Grade this student essay on a scale of A–F with one sentence justification.\n'
    'Topic: {topic}\nEssay:\n{essay}'
)

feedback_prompt = ChatPromptTemplate.from_template(
    'The essay received the following grade:\n{grade}\n\n'
    'Write two specific, encouraging improvement suggestions for the student.'
)

grade_chain = grade_prompt | llm | parser

# Chain grade output into feedback as 'grade' key
full_grading_chain = (
    RunnablePassthrough.assign(grade=grade_chain)
    | feedback_prompt | llm | parser
)

result = full_grading_chain.invoke({
    'topic': 'The impact of AI on modern education',
    'essay': ('AI is transforming classrooms. Students can now access personalised tutors '
              'at any time. However, concerns about academic integrity and over-reliance '
              'on automation remain significant challenges for educators.')
})
print(result)

### 1.3 Parallel Execution — Healthcare: Multi-Angle Patient Report

<small>
`RunnableParallel` runs branches concurrently and collects results as a dict.
Healthcare scenario: analyse a patient discharge summary from three perspectives simultaneously.
</small>

In [ ]:
discharge_summary = '''
Patient: 58-year-old male. Admitted for acute chest pain.
Diagnosed: NSTEMI. Treated with dual antiplatelet therapy.
Discharged after 3 days. Follow-up cardiology appointment in 2 weeks.
Prescribed: aspirin 100mg, ticagrelor 90mg, atorvastatin 40mg.
'''

clinical_prompt   = ChatPromptTemplate.from_template('Summarise the clinical findings:\n{summary}')
medication_prompt = ChatPromptTemplate.from_template('List medication risks to monitor:\n{summary}')
followup_prompt   = ChatPromptTemplate.from_template('What follow-up actions are required?\n{summary}')

parallel_analysis = RunnableParallel(
    clinical_summary  = clinical_prompt   | llm | parser,
    medication_risks  = medication_prompt | llm | parser,
    followup_actions  = followup_prompt   | llm | parser,
)

results = parallel_analysis.invoke({'summary': discharge_summary})
for section, text in results.items():
    print(f'\n=== {section.upper()} ===')
    print(text)

### 1.4 Dynamic Routing — Finance vs Education Queries

<small>
`RunnableBranch` evaluates predicates in order and routes to the first matching branch.
Use this to send finance queries to a finance-tuned prompt and education queries to a different one.
</small>

In [ ]:
finance_prompt = ChatPromptTemplate.from_template(
    'You are a financial advisor. Answer concisely:\n{query}'
)
education_prompt = ChatPromptTemplate.from_template(
    'You are an academic tutor. Answer concisely:\n{query}'
)
healthcare_prompt = ChatPromptTemplate.from_template(
    'You are a medical information assistant. Answer concisely:\n{query}'
)

from langchain_core.runnables import RunnableBranch

router = RunnableBranch(
    (lambda x: any(k in x['query'].lower() for k in ['stock','portfolio','invest','earnings','market']),
     finance_prompt   | llm | parser),
    (lambda x: any(k in x['query'].lower() for k in ['course','study','grade','exam','tutor','student']),
     education_prompt | llm | parser),
    healthcare_prompt | llm | parser,   # default branch
)

queries = [
    {'query': 'How do I diversify my stock portfolio?'},
    {'query': 'What study techniques work best for medical exams?'},
    {'query': 'What are common symptoms of vitamin D deficiency?'},
]

for q in queries:
    print(f'\nQuery : {q["query"]}')
    print(f'Answer: {router.invoke(q)[:200]}')

### 1.5 Memory Management — Education: Adaptive Tutoring Chatbot

<small>
Modern LangChain uses `RunnableWithMessageHistory` instead of deprecated `ConversationBufferMemory`.
Each `session_id` gets its own isolated history — perfect for a multi-student tutoring platform.
</small>

In [ ]:
from C2_4_Func import get_session_history

tutor_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are an adaptive tutor for a finance student. '
               'Remember what the student has told you and build on prior explanations.'),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{input}'),
])

tutor_chain = tutor_prompt | llm

# Session store: production would use Redis or Postgres
session_store = {}

tutoring_app = RunnableWithMessageHistory(
    tutor_chain,
    lambda session_id: get_session_history(session_id, session_store),
    input_messages_key='input',
    history_messages_key='history',
)

student_session = 'student_priya_001'

exchanges = [
    'I am a finance student struggling with DCF valuation.',
    'Can you give me a simple example using a coffee shop?',
    'What discount rate should I use for that coffee shop?',
]

for msg in exchanges:
    print(f'\nStudent: {msg}')
    resp = tutoring_app.invoke(
        {'input': msg},
        config={'configurable': {'session_id': student_session}}
    )
    print(f'Tutor  : {resp.content[:300]}')

### 1.6 Streaming — Real-Time Investment Commentary
<small>Any LCEL chain supports streaming with `.stream()` — tokens arrive as they are generated.</small>

In [ ]:
commentary_prompt = ChatPromptTemplate.from_template(
    'Write a 3-sentence market commentary on: {event}'
)
stream_chain = commentary_prompt | llm | parser

print('Streaming commentary:')
for chunk in stream_chain.stream({'event': 'US Federal Reserve holds rates steady'}):
    print(chunk, end='', flush=True)
print()

---
## Section 2 · LlamaIndex — Data Connectors, Index Types, and Query Engines

### When to Use LlamaIndex vs LangChain

| Dimension | LangChain strength | LlamaIndex strength |
|-----------|-------------------|---------------------|
| **Primary use case** | General LLM orchestration, agents, chains | Document retrieval and Q&A |
| **Data ingestion** | Basic loaders, manual chunking | 100+ connectors, smart chunking |
| **Index types** | VectorStore abstraction | Vector, summary, keyword, knowledge graph |
| **Query strategies** | RAG chain (retrieve → generate) | Sub-question, multi-step, router query |
| **Memory** | `RunnableWithMessageHistory`, LangGraph | Chat engine with built-in memory |
| **Agent tools** | LangChain tools, MCP | QueryEngine as a tool |
| **Best for** | Multi-step workflows, routing, production apps | Deep document understanding, RAG |

**Rule of thumb:** Use LangChain when you need complex orchestration logic.
Use LlamaIndex when your bottleneck is retrieval quality over large document sets.
They compose well — a LlamaIndex query engine can be a LangChain tool.

### 2.1 Setup — Finance Document Corpus (In-Memory)
<small>We create synthetic finance and education documents in memory.
In production, use `SimpleDirectoryReader`, `DatabaseReader`, or any of the 100+ LlamaIndex connectors.</small>

In [ ]:
from llama_index.core import Document, VectorStoreIndex, SummaryIndex, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.anthropic import Anthropic as LlamaAnthropic
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector
from llama_index.core.tools import QueryEngineTool
from pathlib import Path

from C2_4_Func import build_c2_4_corpus, persist_c2_4_corpus, load_c2_4_corpus

Settings.llm = LlamaAnthropic(
    model='claude-haiku-4-5-20251001',
    api_key=ANTHROPIC_API_KEY
)
Settings.node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=64)

# ── Finance + education corpus (persisted to disk, loaded back) ──────────────────────────
CORPUS_DIR = Path('Track2') / 'demos' / 'data' / 'C2.4_corpus'
raw_docs = build_c2_4_corpus()
persist_c2_4_corpus(raw_docs, CORPUS_DIR)
loaded_docs = load_c2_4_corpus(CORPUS_DIR)

finance_docs = [Document(text=d['text'], metadata=d['metadata']) for d in loaded_docs if d['metadata']['domain'] == 'finance']
education_docs = [Document(text=d['text'], metadata=d['metadata']) for d in loaded_docs if d['metadata']['domain'] == 'education']
all_docs = finance_docs + education_docs
print(f'Loaded {len(all_docs)} documents ({len(finance_docs)} finance, {len(education_docs)} education)')

### 2.2 Index Types

<small>
| Index | How it stores | Best for |
|-------|--------------|---------|
| `VectorStoreIndex` | Embedding vectors | Semantic similarity search |
| `SummaryIndex` | All nodes in sequence | Full-document summarisation |
| `KeywordTableIndex` | Keyword → node mapping | Keyword-based lookup |
</small>

In [ ]:
# VectorStoreIndex — best for semantic Q&A
vector_index = VectorStoreIndex.from_documents(finance_docs)

# SummaryIndex — best for 'give me a summary of everything'
summary_index = SummaryIndex.from_documents(education_docs)

print('Indexes built.')
print(f'VectorIndex nodes  : {len(vector_index.docstore.docs)}')
print(f'SummaryIndex nodes : {len(summary_index.docstore.docs)}')

### 2.3 Query Engines

<small>
A **query engine** wraps an index and exposes a `.query()` method.
The router query engine selects between engines automatically based on the question.
</small>

In [ ]:
# Basic vector query engine
vector_qe = vector_index.as_query_engine(similarity_top_k=2)

# Summary query engine
summary_qe = summary_index.as_query_engine(response_mode='tree_summarize')

# Query the finance corpus
q1 = 'What are the main revenue drivers for FinTech Corp?'
r1 = vector_qe.query(q1)
print(f'Q: {q1}')
print(f'A: {r1}\n')

# Query the education corpus
q2 = 'What interventions improved course completion rates?'
r2 = summary_qe.query(q2)
print(f'Q: {q2}')
print(f'A: {r2}')

### 2.4 Router Query Engine
<small>The router uses an LLM to decide which underlying engine to call for each query.</small>

In [ ]:
# Combine all docs into a single index for router demo
combined_index = VectorStoreIndex.from_documents(all_docs)
combined_qe    = combined_index.as_query_engine(similarity_top_k=3)

# Annotate the engine so the router knows what it covers
finance_tool = QueryEngineTool.from_defaults(
    query_engine=vector_qe,
    description='Finance domain: annual reports, earnings, investment theses for companies.',
)
education_tool = QueryEngineTool.from_defaults(
    query_engine=summary_qe,
    description='Education domain: course catalogs, learning outcomes, completion rates.',
)

router_qe = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[finance_tool, education_tool],
)

for question in [
    'What growth rate is EduLearn targeting?',
    'Which course should I take before DS-301?',
]:
    resp = router_qe.query(question)
    print(f'Q: {question}')
    print(f'A: {resp}\n')

---
## Section 3 · MCP — Model Context Protocol

### What is MCP?

<small>
MCP (Model Context Protocol) is an open standard published by Anthropic that defines how AI models
communicate with external tools and data sources.
It replaces ad-hoc API integrations with a **single protocol** — any MCP server can be used by any MCP client.

```
Client (Claude / LangChain agent)  ←──MCP protocol──→  MCP Server (your tools)
```

**Why MCP matters for multi-agent systems:**
- Agents from different vendors can share the same tool servers
- Tool definitions live in one place — no duplication across agents
- Authentication, logging, and rate limiting are handled at the server layer

**MCP server structure:**
```
MCP Server
├── list_tools()       → returns available tool schemas
├── call_tool(name, args) → executes a tool and returns result
├── list_resources()   → exposes data resources (files, databases)
└── list_prompts()     → exposes reusable prompt templates
```
</small>

### 3.1 Building a Local MCP Server
<small>We write a finance + education MCP server and save it to `mcp_c24_server.py`.
The server exposes four tools: stock price lookup, portfolio analysis, course search, and enrollment check.</small>

In [ ]:
MCP_SERVER_CODE = '''
import asyncio, json
from mcp.server.fastmcp import FastMCP

mcp = FastMCP('finance-education-tools')

# ── Mock data ────────────────────────────────────────────────────────────────
STOCK_DATA = {
    'AAPL': {'price': 182.50, 'change_pct': 1.2,  'sector': 'Technology'},
    'JPM' : {'price': 198.75, 'change_pct': -0.3, 'sector': 'Finance'},
    'AMZN': {'price': 175.40, 'change_pct': 0.8,  'sector': 'E-commerce'},
    'UNH' : {'price': 512.30, 'change_pct': -1.1, 'sector': 'Healthcare'},
    'EDU' : {'price': 45.20,  'change_pct': 2.1,  'sector': 'Education'},
}

COURSES = {
    'DS-101': {'title': 'Intro to Python',   'prereqs': [],        'seats_left': 45},
    'DS-201': {'title': 'Stats for DS',      'prereqs': ['DS-101'],'seats_left': 12},
    'DS-301': {'title': 'ML Fundamentals',   'prereqs': ['DS-201'],'seats_left': 8},
    'FIN-201': {'title': 'Corporate Finance','prereqs': [],        'seats_left': 30},
}

# ── Tools ────────────────────────────────────────────────────────────────────
@mcp.tool()
def get_stock_price(ticker: str) -> str:
    \"\"\"Return current price and sector for a stock ticker.\"\"\"
    t = ticker.upper()
    if t not in STOCK_DATA:
        return json.dumps({'error': f'Ticker {t} not found'})
    d = STOCK_DATA[t]
    return json.dumps({'ticker': t, **d})

@mcp.tool()
def get_portfolio_summary(tickers: list[str]) -> str:
    \"\"\"Summarise a portfolio given a list of ticker symbols.\"\"\"
    holdings = []
    for t in tickers:
        d = STOCK_DATA.get(t.upper())
        if d:
            holdings.append({'ticker': t.upper(), **d})
    if not holdings:
        return json.dumps({'error': 'No valid tickers found'})
    sectors = {}
    for h in holdings:
        sectors[h[\'sector\']] = sectors.get(h[\'sector\'], 0) + 1
    return json.dumps({'holdings': holdings, 'sector_weights': sectors})

@mcp.tool()
def search_courses(topic: str, level: str = \'any\') -> str:
    \"\"\"Search available courses by topic keyword.\"\"\"
    matches = []
    for cid, info in COURSES.items():
        if topic.lower() in info[\'title\'].lower() or topic.lower() in cid.lower():
            matches.append({'id': cid, **info})
    return json.dumps({'courses': matches, 'query': topic})

@mcp.tool()
def check_enrollment(course_id: str, student_completed: list[str]) -> str:
    \"\"\"Check if a student is eligible to enroll in a course.\"\"\"
    c = COURSES.get(course_id.upper())
    if not c:
        return json.dumps({'error': f'Course {course_id} not found'})
    missing = [p for p in c[\'prereqs\'] if p not in student_completed]
    eligible = len(missing) == 0
    return json.dumps({
        'course_id': course_id,
        'eligible': eligible,
        'missing_prereqs': missing,
        'seats_left': c[\'seats_left\'],
    })

if __name__ == \'__main__\':
    mcp.run(transport=\'stdio\')
'''

# Save the server to a file
Path('mcp_c24_server.py').write_text(MCP_SERVER_CODE, encoding='utf-8')
print('MCP server saved to mcp_c24_server.py')

### 3.2 MCP Server Structure Explained

<small>
```
@mcp.tool()            ← registers the function as an MCP tool
def get_stock_price(ticker: str) -> str:
    \"\"\"Return current price...\"\"\"   ← docstring becomes the tool description
    ...                ← implementation
```

The `FastMCP` class auto-generates the JSON schema from Python type hints.
The server communicates over **stdio** (default) — the client spawns it as a subprocess.
For network deployment, use `transport='sse'` or `transport='http'`.
</small>

### 3.3 Connecting to the MCP Server

<small>
The MCP Python SDK provides `stdio_client` to connect to a server subprocess.
The client:
1. Spawns the server process
2. Calls `session.initialize()` to handshake
3. Uses `session.list_tools()` and `session.call_tool()` to interact
</small>

In [ ]:
from C2_4_Func import demo_mcp_client

await demo_mcp_client('mcp_c24_server.py', sys.executable)

### 3.4 Claude Agent Using MCP Tools
<small>
We pass the MCP tool schemas directly to the Anthropic API.
The agent decides which tools to call; we execute them on the MCP server.
</small>

In [ ]:
from C2_4_Func import run_mcp_agent

# Finance scenario
print('=== Finance Query ===')
await run_mcp_agent(
    'My portfolio has AAPL, JPM, and UNH. Am I too concentrated in any sector?',
    ANTHROPIC_API_KEY, 'mcp_c24_server.py', sys.executable
)

# Education scenario
print('\n=== Education Query ===')
await run_mcp_agent(
    'I have completed DS-101. Can I enroll in DS-301?',
    ANTHROPIC_API_KEY, 'mcp_c24_server.py', sys.executable
)

---
## Section 4 · Multi-Agent Orchestration — Coordinator + Specialist Pattern

<small>
**When is multi-agent complexity justified?**

| Situation | Recommendation |
|-----------|---------------|
| Single domain, single task | Use one agent or a chain |
| Multiple domains, independent tasks | Multi-agent: one specialist per domain |
| Tasks that depend on each other | Multi-agent with explicit context passing |
| Parallel independent subtasks | Multi-agent (run specialists concurrently) |

**The coordinator + specialist pattern:**
- **Coordinator** receives the user request, decides which specialists to activate
- **Specialists** are domain-expert agents that return structured results
- **Coordinator** synthesises specialist outputs into a final response

**Keep to two agents for most use cases.** A third agent usually means the coordinator
can be replaced with routing logic.
</small>

In [ ]:
from C2_4_Func import finance_specialist, education_specialist
import anthropic

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

print('Specialist agents ready.')

In [ ]:
from C2_4_Func import run_coordinator

# ── Coordinator agent ────────────────────────────────────────────────────────

COORDINATOR_TOOLS = [
    {
        'name': 'call_finance_specialist',
        'description': 'Delegate a finance-related sub-question to the finance specialist. '
                       'Provide any relevant financial data in the context field.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'sub_query': {'type': 'string', 'description': 'The specific finance question'},
                'context':   {'type': 'object',  'description': 'Relevant data to pass to the specialist'},
            },
            'required': ['sub_query', 'context'],
        },
    },
    {
        'name': 'call_education_specialist',
        'description': 'Delegate an education or course-planning sub-question to the education specialist.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'sub_query': {'type': 'string'},
                'context':   {'type': 'object'},
            },
            'required': ['sub_query', 'context'],
        },
    },
]

COORDINATOR_SYSTEM = '''You are a coordinator agent for a professional learning and investment platform.
Your users are finance professionals who also want to upskill via courses.

When a user sends a query:
1. Decide which specialist(s) can best answer it.
2. Call the relevant specialist(s) with a precise sub-query and any useful context.
3. Synthesise their answers into a concise, actionable response for the user.

Do not guess — if you need financial or educational data, ask the right specialist.
'''

print('Coordinator agent ready.')

In [ ]:
# ── Demo: user is a finance professional who wants to expand into AI ─────────

shared_context = {
    'user_profile': {
        'role': 'Portfolio Manager',
        'completed_courses': ['DS-101'],
        'portfolio_tickers': ['AAPL', 'JPM', 'UNH'],
        'risk_tolerance': 'moderate',
    }
}

query = (
    'Given my portfolio and my DS-101 background, '
    'should I invest in healthcare AI stocks '
    'and which course should I take next to better evaluate AI companies?'
)

print(f'User query: {query}\n')
final_answer, log = run_coordinator(
    query, shared_context, client, COORDINATOR_TOOLS, COORDINATOR_SYSTEM,
    lambda q, c: finance_specialist(q, c, client),
    lambda q, c: education_specialist(q, c, client),
)

print('=== Coordinator log ===')
for entry in log:
    print(f"  Step {entry['step']}: delegated to {entry['specialist']} specialist")
    print(f"    Sub-query: {entry['query']}")

print(f'\n=== Final answer ===\n{final_answer}')

---
## Section 5 · Prompt Versioning and A/B Testing

<small>
In production, prompts are as important as code.
A prompt change that improves accuracy by 5% can be worth more than a week of engineering.

**Why version prompts?**
- Roll back quickly if a new prompt regresses
- Track which prompt version produced which output
- Run controlled A/B tests with production traffic
- Audit trail for regulated industries (finance, healthcare)

**Prompt registry pattern:**
- Central registry maps `(name, version)` → template
- Each deployment pins a version
- A/B test by randomly assigning users to version A or B
- Measure: output quality score, user satisfaction, task completion rate
</small>

In [ ]:
from C2_4_Func import PromptVersion, PromptRegistry

registry = PromptRegistry()
print('PromptRegistry ready.')

In [ ]:
# ── Finance: Loan Approval Decision Prompt — two versions ───────────────────

registry.register(PromptVersion(
    name='loan_decision',
    version='v1',
    template=(
        'Evaluate this loan application and respond with APPROVE or DENY.\n'
        'Application: {application}'
    ),
    description='Baseline prompt — simple approve/deny',
    author='risk-team',
    tags=['finance', 'credit'],
))

registry.register(PromptVersion(
    name='loan_decision',
    version='v2',
    template=(
        'You are a senior credit analyst. Evaluate the following loan application.\n'
        'Respond with a JSON object containing:\n'
        '- decision: APPROVE or DENY\n'
        '- risk_score: 1-10 (10 = highest risk)\n'
        '- key_factors: list of up to 3 decisive factors\n'
        '- conditions: any conditions attached to approval (empty list if none)\n\n'
        'Application:\n{application}'
    ),
    description='Structured JSON output with risk scoring',
    author='risk-team',
    tags=['finance', 'credit', 'structured-output'],
))

# ── Education: Course Recommendation Prompt — two versions ───────────────────

registry.register(PromptVersion(
    name='course_recommendation',
    version='v1',
    template='Recommend a course for: {student_profile}',
    description='Simple recommendation',
    author='edu-team',
))

registry.register(PromptVersion(
    name='course_recommendation',
    version='v2',
    template=(
        'You are an academic advisor.\n'
        'Student profile: {student_profile}\n'
        'Available courses: {available_courses}\n\n'
        'Recommend the single best next course. Explain why in 2 sentences.',
    ),
    description='Contextual recommendation with available courses',
    author='edu-team',
))

print('All registered prompts:', registry.all_prompts())
for name in registry.all_prompts():
    print(f'  {name}: versions = {registry.list_versions(name)}')

### 5.1 A/B Testing Framework
<small>
We randomly assign each request to version A or version B, then track quality scores.
After N requests, compare average scores to decide which version wins.
In production, use user-segment based routing (e.g., 10% traffic to v2) for safer rollouts.
</small>

In [ ]:
from C2_4_Func import ABTest, run_coordinator, score_loan_response

# ── Loan decision A/B test ───────────────────────────────────────────────────

loan_applications = [
    {'id': 'req-001', 'app': 'Annual income $85,000. Loan amount $200,000. Credit score 740. Employed 4 years.'},
    {'id': 'req-002', 'app': 'Annual income $32,000. Loan amount $150,000. Credit score 580. Self-employed 1 year.'},
    {'id': 'req-003', 'app': 'Annual income $120,000. Loan amount $250,000. Credit score 810. Employed 10 years.'},
    {'id': 'req-004', 'app': 'Annual income $55,000. Loan amount $180,000. Credit score 650. Employed 2 years.'},
]

ab_test = ABTest(client, registry, 'loan_decision', 'v1', 'v2', traffic_split=0.5)

print('Running A/B test on loan_decision prompt...\n')
for item in loan_applications:
    result = ab_test.run(
        request_id=item['id'],
        template_vars={'application': item['app']},
        score_fn=score_loan_response
    )
    print(f"  [{result['request_id']}] version={result['version']} score={result['score']}")
    print(f"  output: {result['output'][:120]}...\n")

print('=== A/B Test Summary ===')
print(json.dumps(ab_test.summary(), indent=2))